# Chapter 1 — What Context Means

## Question

**If information exists somewhere in the system, does the model actually have it?**

Falsifiable version: holding available information constant, does changing the admission decision change the current context? If yes, then `available \u2260 session state \u2260 current context \u2260 window capacity`.

No model is called here. Deliberate: Chapter 1 labels the behavioural effect of changing context a *book hypothesis*, not a result. This notebook establishes the objects later chapters experiment on.

## Setup — a small synthetic evening

A developer asks: *the checkout test fails after my last commit, please fix it.* Token counts are illustrative fixtures, not measurements.

In [ ]:
from dataclasses import dataclass

@dataclass
class InformationItem:
    id: str
    label: str
    kind: str
    text: str
    tokens: int
    available: bool = True
    persisted: bool = False
    admitted: bool = False

@dataclass
class ContextBundle:
    items: list
    rendered_text: str
    input_tokens: int

items = [
    InformationItem('system', 'System instructions', 'instructions', 'You are a coding agent. Follow project rules.', 500, True, True, True),
    InformationItem('user_request', 'User request', 'request', 'the checkout test fails after my last commit, please fix it.', 15, True, True, True),
    InformationItem('checkout_py', 'checkout.py (excerpt)', 'file', '<contents of checkout.py as supplied this turn>', 900, True, False, True),
    InformationItem('ci_failure', 'CI failure log', 'log', '<failing assertion output from CI>', 650, True, True, True),
    InformationItem('previous_plan', 'Previous plan note', 'note', 'Plan: reproduce, isolate commit, run checkout test.', 180, True, True, True),
    InformationItem('readme', 'README.md', 'file', '<project readme; not supplied this turn>', 800, True, False, False),
    InformationItem('old_error', 'Old error trace', 'log', '<trace pasted three turns ago; stored, not re-supplied>', 1100, True, True, False),
    InformationItem('search_2', 'Search result 2', 'tool-output', '<second search result; stored>', 400, True, True, False),
    InformationItem('env', 'environment.json', 'config', '<env config; stored>', 250, True, True, False),
    InformationItem('git_history', 'Git history', 'history', '<full log; on disk, never loaded>', 5000, True, False, False),
    InformationItem('deployment', 'deployment.md', 'file', 'Production migrations require approval.', 120, True, False, False),
]
print(f'{len(items)} items defined.')

```text
AVAILABLE INFORMATION: everything the system could access
  SESSION STATE: the persisted subset (survives between invocations)
    CURRENT CONTEXT: the admitted subset (this invocation)
CONTEXT WINDOW: not a set — a capacity constraint.
```

In [ ]:
avail = [x for x in items if x.available]
print(f'AVAILABLE: {len(avail)} items, {sum(x.tokens for x in avail)} fixture tokens')
for x in avail:
    print(f"  {x.id:14s} {x.tokens:5d}  persisted={x.persisted!s:5s} admitted={x.admitted!s:5s}  {x.label}")

In [ ]:
state = [x for x in items if x.persisted]
print(f'SESSION STATE: {len(state)} items, {sum(x.tokens for x in state)} fixture tokens')
for x in state:
    print(f"  {x.id:14s} {x.tokens:5d}  admitted={x.admitted!s:5s}  {x.label}")
print()
print('Persistence does not imply visibility.')

## Baseline — render the current context

In [ ]:
def render_context(all_items, order=None):
    admitted = [x for x in all_items if x.admitted]
    if order is not None:
        pos = {cid: i for i, cid in enumerate(order)}
        admitted = sorted(admitted, key=lambda x: pos.get(x.id, 999))
    parts = ['--- ' + x.label + ' (' + str(x.tokens) + ' tokens) ---\n' + x.text for x in admitted]
    rendered = '\n\n'.join(parts)
    return ContextBundle(items=admitted, rendered_text=rendered, input_tokens=sum(x.tokens for x in admitted))

bundle = render_context(items)
print(bundle.rendered_text)
print()
print(f'CURRENT CONTEXT: {len(bundle.items)} items, {bundle.input_tokens} tokens')

In [ ]:
context_window = 3000
reserved_output = 500
usable_input = context_window - reserved_output
print(f'advertised capacity       {context_window}')
print(f'reserved output            {reserved_output}')
print(f'usable input capacity     {usable_input}')
print(f'current rendered context  {bundle.input_tokens}')
print(f'remaining                   {usable_input - bundle.input_tokens}')

## Intervention 1 — change admission, hold everything else fixed

In [ ]:
def show_context(all_items, window=3000, reserved=500):
    b = render_context(all_items)
    avail_n = sum(1 for x in all_items if x.available)
    state_n = sum(1 for x in all_items if x.persisted)
    print(f'AVAILABLE INFORMATION: unchanged ({avail_n} items)')
    print(f'SESSION STATE:         unchanged ({state_n} persisted)')
    print(f'CONTEXT:               changed  ({len(b.items)} items, {b.input_tokens} tokens)')
    print(f'WINDOW CAPACITY:       unchanged ({window}, {reserved} reserved)')
    return b

def admit(all_items, item_id):
    next(x for x in all_items if x.id == item_id).admitted = True

def exclude(all_items, item_id):
    next(x for x in all_items if x.id == item_id).admitted = False

admit(items, 'readme')
b1 = show_context(items)
print('Admitted:', [x.id for x in b1.items])
print()
exclude(items, 'ci_failure')
b2 = show_context(items)
print('Admitted:', [x.id for x in b2.items])
exclude(items, 'readme')
admit(items, 'ci_failure')

## Intervention 2 — change representation, hold the fact fixed

In [ ]:
ci = next(x for x in items if x.id == 'ci_failure')
print(f'Before: {ci.tokens} tokens')
saved_tokens, saved_text = ci.tokens, ci.text
ci.tokens, ci.text = 70, 'FAIL test_total: expected 42.00, got 41.50 (excerpt)'
b3 = render_context(items)
print(f'After: {ci.tokens} tokens')
print(f'Context total: {bundle.input_tokens} -> {b3.input_tokens} tokens (same fact, different representation)')
ci.tokens, ci.text = saved_tokens, saved_text

## Observation — the Context Report

In [ ]:
bundle = render_context(items)
avail_t = sum(x.tokens for x in items if x.available)
state_items = [x for x in items if x.persisted]
state_t = sum(x.tokens for x in state_items)
print('INFORMATION UNIVERSE')
print(f'{len(items)} items, {avail_t} tokens potentially available')
print()
print('SESSION STATE')
print(f'{len(state_items)} persisted items, {state_t} tokens')
print()
print('CURRENT CONTEXT')
print(f'{len(bundle.items)} admitted items, {bundle.input_tokens} tokens')
print()
print('CONTEXT WINDOW: 3000 capacity, 500 reserved, 2500 usable')
print()
print('NOT CURRENT CONTEXT')
for x in items:
    if not x.admitted:
        where = 'persisted, not admitted' if x.persisted else 'available, not admitted'
        print(f'  {x.label}: {where}')
print()
print('Model knows README.md? No. System can access it? Yes.')
print('System remembers old_error? Yes. Can it influence this computation? No.')
print('Window changed on admit? No. Context changed? Yes.')

## Deliberate trap — \u201cbut it had access to the repo\u201d

> `deployment.md` says production migrations require approval. Is that rule part of the model's context?

In [ ]:
deployment = next(x for x in items if x.id == 'deployment')
print('available:', deployment.available)
print('admitted: ', deployment.admitted)
print('Therefore: available is not context.')

## Try it

1. `exclude(items, 'ci_failure'); show_context(items)`
2. `admit(items, 'readme'); show_context(items)` — watch the 2500-token usable budget.
3. `admit(items, 'git_history'); show_context(items)` — then `exclude(items, 'git_history')` to restore.
4. Re-run the Context Report cell after each change.

Restore baseline: admitted = system, user_request, checkout_py, ci_failure, previous_plan.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# exclude(items, 'ci_failure')
# admit(items, 'readme')
# show_context(items)

## Next: an unexecuted experiment (hands over to Chapter 2)

We can construct two bundles but cannot yet claim what the model received or whether the difference mattered.

- Bundle A: task + failing file (user_request, checkout_py).
- Bundle B: same + one plausible but unnecessary document (readme).
- Hold fixed: model, version, decoding settings, task.
- Pre-register: context mattered only if outcome distributions differ beyond trial variation, or tool-call traces diverge.

Chapter 2 asks: how do we see the whole rendered thing, token by token?

In [ ]:
bundle_a_ids = ['system', 'user_request', 'checkout_py']
bundle_b_ids = ['system', 'user_request', 'checkout_py', 'readme']
for name, ids in [('Bundle A', bundle_a_ids), ('Bundle B', bundle_b_ids)]:
    toks = sum(next(x for x in items if x.id == cid).tokens for cid in ids)
    print(f'{name}: {ids} ({toks} fixture tokens)')
print('NOT RUN: no model calls, no outcomes. Construction only.')

## What this demonstrates

- Context is the admitted, ordered, rendered bundle for one computation.
- Admission can change while available information and session state stay fixed.
- Representation can change while the underlying fact stays fixed.
- The window bounds the invocation; it does not select contents.

## What this does not demonstrate

- That any bundle is better: no model ran, nothing was measured.
- That fewer tokens cost less, the excerpt preserves what matters, or admission helped. A token total is not cost; a fluent excerpt is not preserved information; an admitted document is not improved behaviour.
- Real provider accounting. Counts are fixtures; Chapter 4 owns the budget.

## Connection to the chapter

The four-way separation in executable form. Every later notebook operates on the `ContextBundle` built here. Next: the instrument that shows what the model actually received.